# Level 0 WWPGD multi-seed diagnostics

Mean and standard-deviation bands are descriptive; use paired confirmation seeds for optimizer claims.


In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(os.getenv('NANOGPT_LEVEL0_WWPGD_RESULTS_ROOT', '/tmp/nanogpt-level0-wwpgd/results'))
rows = []
selected_rows = []
for path in sorted(ROOT.glob('*_seed_*/metrics.csv')):
    optimizer, seed_text = path.parent.name.rsplit('_seed_', 1)
    frame = pd.read_csv(path)
    frame['optimizer'] = optimizer
    frame['seed'] = int(seed_text)
    rows.append(frame)
    selected_path = path.parent / 'selected_checkpoint_metrics.json'
    if selected_path.exists():
        selected = json.loads(selected_path.read_text())
        selected.update({'optimizer': optimizer, 'seed': int(seed_text)})
        selected_rows.append(selected)
all_df = pd.concat(rows, ignore_index=True)
selected_df = pd.DataFrame(selected_rows)
all_df.groupby('optimizer').seed.nunique(), selected_df


In [ ]:
def band_plot(metric, ylabel=None):
    fig, ax = plt.subplots(figsize=(10, 5))
    for optimizer, data in all_df.groupby('optimizer'):
        aggregate = data.groupby('step')[metric].agg(['mean','std']).reset_index()
        std = aggregate['std'].fillna(0)
        line, = ax.plot(aggregate.step, aggregate['mean'], label=optimizer)
        ax.fill_between(aggregate.step, aggregate['mean']-std, aggregate['mean']+std, alpha=.2, color=line.get_color())
    ax.set(xlabel='optimizer step', ylabel=ylabel or metric, title=f'{metric}: mean +/- 1 standard deviation')
    ax.grid(alpha=.25); ax.legend(); plt.show()

for metric in ['val_loss','val_perplexity','val_accuracy','val_generalization_gap']:
    band_plot(metric, 'accuracy fraction' if metric == 'val_accuracy' else metric)


In [ ]:
ww_rows = []
for run in sorted(ROOT.glob('*_seed_*')):
    optimizer, seed_text = run.name.rsplit('_seed_', 1)
    for path in run.glob('weightwatcher_step_*.csv'):
        frame = pd.read_csv(path)
        frame['optimizer'] = optimizer
        frame['seed'] = int(seed_text)
        ww_rows.append(frame)
if ww_rows:
    ww = pd.concat(ww_rows, ignore_index=True)
    ww['alpha'] = pd.to_numeric(ww['alpha'], errors='coerce')
    ww = ww[np.isfinite(ww.alpha)]
    for matrix_type, matrix_data in ww.groupby('matrix_type'):
        fig, ax = plt.subplots(figsize=(10, 4))
        for optimizer, optimizer_data in matrix_data.groupby('optimizer'):
            per_seed = optimizer_data.groupby(['seed','step']).alpha.mean().reset_index()
            aggregate = per_seed.groupby('step').alpha.agg(['mean','std']).reset_index()
            std = aggregate['std'].fillna(0)
            line, = ax.plot(aggregate.step, aggregate['mean'], label=optimizer)
            ax.fill_between(aggregate.step, aggregate['mean']-std, aggregate['mean']+std, alpha=.2, color=line.get_color())
        ax.axhline(2.0, linestyle='--', linewidth=1)
        ax.set(xlabel='optimizer step', ylabel='WeightWatcher alpha', title=matrix_type)
        ax.grid(alpha=.25); ax.legend(); plt.show()


In [ ]:
# All matrix alpha trajectories on one plot with cross-seed bands.
BAND_SIGMA = 1.0
per_seed = ww.groupby(['optimizer','matrix_name','matrix_type','block','seed','step'], as_index=False).alpha.mean()
summary = per_seed.groupby(['optimizer','matrix_name','matrix_type','block','step'], as_index=False).alpha.agg(mean_alpha='mean', std_alpha='std', seed_count='count')
summary['std_alpha'] = summary.std_alpha.fillna(0.0)
series = summary[['optimizer','matrix_name','matrix_type','block']].drop_duplicates().sort_values(['optimizer','block','matrix_type','matrix_name']).reset_index(drop=True)
colors = plt.cm.turbo(np.linspace(0.02, 0.98, len(series)))
fig, ax = plt.subplots(figsize=(18, 10))
for color, meta in zip(colors, series.itertuples(index=False)):
    trajectory = summary[(summary.optimizer == meta.optimizer) & (summary.matrix_name == meta.matrix_name)].sort_values('step')
    mean = trajectory.mean_alpha.to_numpy(float); std = trajectory.std_alpha.to_numpy(float); steps = trajectory.step.to_numpy(float)
    label = f'{meta.optimizer} | B{int(meta.block)} | {meta.matrix_type}'
    ax.plot(steps, mean, color=color, linewidth=1.8, alpha=.95, label=label)
    ax.fill_between(steps, mean-BAND_SIGMA*std, mean+BAND_SIGMA*std, color=color, alpha=.10, linewidth=0)
ax.axhline(2.0, color='black', linestyle='--', linewidth=1.5, label='target alpha = 2')
ax.set(xlabel='optimizer step', ylabel='WeightWatcher alpha', title=f'All transformer-matrix alpha trajectories: mean +/- {BAND_SIGMA:g} std')
ax.grid(alpha=.20); ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=7)
fig.tight_layout(); plt.show()


In [ ]:
projection_rows = []
for run in sorted(ROOT.glob('*_wwpgd_seed_*')):
    seed = int(run.name.rsplit('_seed_', 1)[1])
    frame = pd.read_csv(run / 'wwpgd_projection.csv'); frame['seed'] = seed; projection_rows.append(frame)
projection = pd.concat(projection_rows, ignore_index=True)
projection['relative_frobenius_change_applied'] = pd.to_numeric(projection['relative_frobenius_change_applied'], errors='coerce')
per_seed_dose = projection.groupby(['seed','optimizer_step']).relative_frobenius_change_applied.mean().reset_index()
dose = per_seed_dose.groupby('optimizer_step').relative_frobenius_change_applied.agg(['mean','std']).reset_index(); dose['std'] = dose['std'].fillna(0)
fig, ax = plt.subplots(figsize=(11, 5)); ax.plot(dose.optimizer_step, dose['mean'])
ax.fill_between(dose.optimizer_step, dose['mean']-dose['std'], dose['mean']+dose['std'], alpha=.2)
ax.set(xlabel='optimizer step', ylabel='mean applied relative matrix change', title='WWPGD dose: mean +/- 1 std across seeds')
ax.grid(alpha=.25); plt.show()
